# Helmet runtime: cùng code trên máy local, Colab và Kaggle

Notebook chỉ thiết lập môi trường và gọi CLI trong repository; không định nghĩa pipeline khác, không tự huấn luyện. Phần benchmark đo thiết bị/FPS thực tế; phần nhận diện ở cuối xử lý ảnh/video mới trên chính GPU Colab. ZIP benchmark không kết nối GPU vào app local. Xem `docs/cloud_workflow_2026-11-05.md`.

Colab: chọn GPU trong Runtime → Change runtime type. Kaggle: bật GPU và Internet khi cần tải package/model; thêm ZIP code vào private input dataset. GPU miễn phí phụ thuộc quota/tình trạng dịch vụ, không được đảm bảo. Dùng notebook tương tác, không dựng web service/tunnel/Gradio trên free Colab. [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Kaggle image source](https://github.com/Kaggle/docker-python).


## 1. Đưa code hiện tại lên cloud

Trên local chạy `python scripts/package_cloud_workspace.py --include-probe`, rồi upload `outputs/cloud/helmet_workspace.zip`. Flag này chỉ thêm probe nhỏ và provenance; không gửi dữ liệu, secret hay checkpoint gated. Bỏ flag nếu muốn tự cung cấp probe. Chỉ dùng ZIP/probe do bạn tin cậy.

Colab sẽ mở hộp upload nếu chưa có thư mục project. Kaggle: đặt `PROJECT_ZIP` thành đường dẫn thật như `/kaggle/input/<your-private-dataset>/helmet_workspace.zip`. Có thể đặt `PROJECT_DIR` tới code hiện tại trong thư mục ghi được. Không clone một remote giả định; ZIP chứa cả thay đổi chưa commit.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

IS_KAGGLE = Path("/kaggle/working").is_dir()
BASE = Path("/kaggle/working" if IS_KAGGLE else "/content" if Path("/content").is_dir() else Path.cwd())
PROJECT_DIR = ""  # Optional: existing writable repository path.
PROJECT_ZIP = ""  # Kaggle: set the actual private input dataset ZIP path.
PROJECT = Path(PROJECT_DIR).expanduser().resolve() if PROJECT_DIR else BASE / "helmet_runtime"

if not (PROJECT / "scripts/benchmark_runtime.py").is_file():
    archive_path = Path(PROJECT_ZIP).expanduser() if PROJECT_ZIP else None
    if archive_path is None:
        if IS_KAGGLE:
            raise RuntimeError("Set PROJECT_ZIP to /kaggle/input/<your-private-dataset>/helmet_workspace.zip.")
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError("Set PROJECT_ZIP or PROJECT_DIR for this environment.") from None
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(zip_names) != 1:
            raise RuntimeError("Upload exactly one trusted project ZIP.")
        archive_path = Path(zip_names[0]).resolve()
        del uploaded
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for entry in archive.infolist():
            target = (PROJECT / entry.filename).resolve()
            if not target.is_relative_to(PROJECT.resolve()) or (entry.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Unsafe ZIP member.")
        archive.extractall(PROJECT)

required = ["scripts/benchmark_runtime.py", "scripts/prepare_benchmark_data.py", "configs/runtime_profiles.json"]
for relative in required:
    if not (PROJECT / relative).is_file():
        raise FileNotFoundError("Repackage current source: missing " + relative)
os.chdir(PROJECT)
print("Project:", PROJECT)

def run_python(*arguments):
    return subprocess.run([sys.executable, *map(str, arguments)], cwd=PROJECT, check=True)


## 2. Kiểm tra GPU trước khi tải model

Ghi nhận CUDA/cuDNN của **PyTorch đang cài**, không suy diễn từ cột “CUDA Version” của driver. Chọn ORT theo [bảng tương thích chính thức](https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html): CUDA 12.8+ / cuDNN 9 dùng pin 1.22.0 từ workflow Kaggle trước; CUDA 12 thấp hơn / cuDNN 9 dùng 1.20.1. Stack khác dừng để chọn wheel đúng bảng; không cài latest tùy ý.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable. Enable a GPU runtime; cloud_cuda will not fall back to CPU.")
cuda_version = tuple(map(int, torch.version.cuda.split(".")[:2]))
cudnn_version = torch.backends.cudnn.version()
if cuda_version[0] != 12 or not cudnn_version or not 90000 <= cudnn_version < 100000:
    raise RuntimeError(f"Review official ORT matrix for this stack: Torch CUDA={torch.version.cuda}, cuDNN={cudnn_version}.")
ORT_VERSION = "1.22.0" if cuda_version >= (12, 8) else "1.20.1"
print({"torch": torch.__version__, "torch_cuda": torch.version.cuda,
       "cudnn": cudnn_version, "gpu": torch.cuda.get_device_name(0),
       "vram_GiB": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
       "selected_onnxruntime_gpu": ORT_VERSION})
subprocess.run(["nvidia-smi"], check=True)


## 3. Cài dependencies, giữ GPU PyTorch của dịch vụ

Cell này ràng buộc phiên bản torch/torchvision đang có, loại gói ORT CPU khỏi requirements và thay bằng ORT GPU đã chọn. Nếu probe có provenance, dùng cùng phiên bản scikit-learn đã serialize probe. Cài thêm package telemetry tùy chọn `nvidia-ml-py` để benchmark lấy VRAM theo process qua NVML, bao gồm ORT; nếu NVML không dùng được, metric này phải ghi unavailable, không coi Torch-only là tổng VRAM. Không cài `requirements-local.txt` cho GPU cloud.

**Sau khi cell hoàn tất, restart kernel/session Python để loại module đã import, chạy lại cell project (mục 1), rồi bỏ qua mục 2–3 và tiếp tục mục 4.** Trên Kaggle dùng restart session Python nếu UI hỗ trợ; không factory-reset toàn runtime hoặc xóa file.


In [ ]:
from importlib import metadata
import json
import tempfile
from packaging.requirements import Requirement

constraints = []
for package in ("torch", "torchvision", "torchaudio"):
    try:
        constraints.append(f"{package}=={metadata.version(package)}")
    except metadata.PackageNotFoundError:
        pass
requirements = ["nvidia-ml-py"]  # Optional NVML telemetry for process VRAM, including ORT.
for line in (PROJECT / "requirements.txt").read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    requirement = Requirement(line)
    if requirement.name.lower().replace("_", "-") not in {"torch", "torchvision", "torchaudio", "onnxruntime", "onnxruntime-gpu"}:
        requirements.append(line)
provenance_path = PROJECT / "weights/dino_probe_provenance.json"
if provenance_path.is_file():
    provenance = json.loads(provenance_path.read_text())
    sklearn_version = provenance.get("scikit_learn_version")
    if sklearn_version:
        constraints.append(f"scikit-learn=={sklearn_version}")
with tempfile.TemporaryDirectory() as temp:
    constraint_path = Path(temp) / "constraints.txt"
    constraint_path.write_text("\n".join(constraints) + "\n")
    run_python("-m", "pip", "install", "-q", "-c", constraint_path, *requirements)
run_python("-m", "pip", "uninstall", "-y", "onnxruntime", "onnxruntime-gpu")
run_python("-m", "pip", "install", "-q", f"onnxruntime-gpu=={ORT_VERSION}")
print("Dependencies installed. Restart Python, rerun project cell only, then continue at section 4.")


## 4. Provider GPU, secrets và checkpoint

`HF_TOKEN` chỉ cần cho lần tải DINOv3 gated; chấp thuận quyền model trên Hugging Face bằng tài khoản của bạn trước. Cell đọc secret qua Colab `userdata` hoặc Kaggle `UserSecretsClient`, không in token. Nếu đã stage model được cấp phép, đặt `DINO_LOCAL_DIR` tới thư mục cha chứa `dinov3-vits16-pretrain-lvd1689m/`; dữ liệu gated chỉ để private. Không upload thư mục cache gated vào output công khai.


In [ ]:
import json
import torch  # Import before ORT to load matching CUDA/cuDNN libraries.
import onnxruntime as ort

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU available.")
if hasattr(ort, "preload_dlls"):
    ort.preload_dlls()
if "CUDAExecutionProvider" not in ort.get_available_providers():
    raise RuntimeError("ORT CUDAExecutionProvider unavailable; fix installation before benchmark.")
LOCAL_DINO_PARENT = ""  # Optional private, licensed model directory on this runtime.
if LOCAL_DINO_PARENT:
    os.environ["DINO_LOCAL_DIR"] = str(Path(LOCAL_DINO_PARENT).expanduser().resolve())
local_dino = Path(os.environ.get("DINO_LOCAL_DIR", "/nonexistent")) / "dinov3-vits16-pretrain-lvd1689m"
if not local_dino.is_dir() and not os.environ.get("HF_TOKEN"):
    try:
        if IS_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        else:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        if not token:
            raise ValueError("Missing token")
        os.environ["HF_TOKEN"] = token
        del token
    except Exception:
        raise RuntimeError("Add an authorized HF_TOKEN secret or stage a private DINO model; token values are never printed.") from None

probe = PROJECT / "weights/dino_probe.joblib"
provenance_path = PROJECT / "weights/dino_probe_provenance.json"
if not probe.is_file() or not provenance_path.is_file():
    raise FileNotFoundError("Supply trusted weights/dino_probe.joblib plus provenance, or repackage with --include-probe.")
import hashlib
provenance = json.loads(provenance_path.read_text())
expected = provenance.get("probe_sha256")
if not expected or hashlib.sha256(probe.read_bytes()).hexdigest() != expected:
    raise RuntimeError("Probe checksum does not match provenance.")
print({"torch": torch.__version__, "torch_cuda": torch.version.cuda,
       "cudnn": torch.backends.cudnn.version(), "onnxruntime": ort.__version__,
       "providers": ort.get_available_providers(), "gpu": torch.cuda.get_device_name(0),
       "vram_GiB": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)})


## 5. Cùng trọng số và held-out manifest

CLI tải model inference có sẵn; không fit model. `prepare_accuracy_models.py` tải head ONNX + CLIP, còn loader YOLO sẽ tải checkpoint chuẩn nếu thiếu. CLI dữ liệu chọn official test clips, tách khỏi training probe. Mẫu 4 × 4 chỉ để benchmark đầu tiên; không đủ cho kết luận accuracy deployment.

Để so sánh local/cloud, dùng **cùng manifest, SHA256 frame/model/probe, config, precision và nhãn**. Upload riêng manifest cùng file tương ứng nếu dùng bộ benchmark local đã khóa. Không trộn dữ liệu training vào manifest test.


In [ ]:
run_python("scripts/prepare_accuracy_models.py")
MANIFEST = Path("data/benchmark/manifest.json")
if not MANIFEST.is_file():
    run_python("scripts/prepare_benchmark_data.py", "--clips", "4", "--frames-per-clip", "4")
if not MANIFEST.is_file():
    raise FileNotFoundError(MANIFEST)
print("Benchmark manifest:", MANIFEST)


## 6. Benchmark, không thay thuật toán theo môi trường

Gọi cùng `scripts/benchmark_runtime.py` với profile trong `configs/runtime_profiles.json`. Local có `baseline_cpu`, `optimized_cpu`, `openvino_cpu`; ở đây chạy lần lượt `baseline_cuda` và `cloud_cuda` trên cùng manifest và cùng GPU. Hai profile giữ cùng device/thread; baseline tắt batch head và cache text, cloud bật hai tối ưu đó. Mặc định mỗi profile chạy 2 repeat; mẫu 4 × 4 frame vẫn chỉ là smoke benchmark. So sánh hai profile CUDA để đo ảnh hưởng tối ưu trên cùng phần cứng; so local CPU với cloud GPU chỉ phản ánh khác biệt cấu hình/phần cứng. CLI chịu trách nhiệm kiểm tra CUDA/provider thực tế, warmup và báo metric. Nếu lỗi, sửa môi trường hoặc model; không tự chuyển sang CPU rồi ghi nhãn GPU. Theo dõi thời gian startup/download riêng khỏi inference và kiểm tra điều kiện accuracy trước khi chọn profile nhanh hơn.


In [ ]:
OUTPUT = Path("outputs/benchmarks/cloud_gpu_run")
for profile in ("baseline_cuda", "cloud_cuda"):
    run_python("scripts/benchmark_runtime.py", "--profile", profile,
               "--manifest", MANIFEST, "--repeats", "2", "--out", OUTPUT / profile)
print("Saved both GPU benchmark profiles:", OUTPUT)
print("Free/total VRAM bytes after benchmark:", torch.cuda.mem_get_info())


## 7. Lưu kết quả trước khi runtime hết hạn

Export thư mục `outputs/benchmarks/cloud_gpu_run/` chứa cả `baseline_cuda/` và `cloud_cuda/`; không export workspace, HF cache, environment variables hay checkpoint gated. Artifact có thể chứa frame dự đoán; kiểm tra quyền chia sẻ dữ liệu trước khi publish. Notebook không tự publish hoặc tạo dịch vụ trả phí.


In [ ]:
import shutil

if not OUTPUT.is_dir():
    raise FileNotFoundError(OUTPUT)
result_archive = Path(shutil.make_archive(str(BASE / "helmet_runtime_benchmark"), "zip", OUTPUT))
if IS_KAGGLE:
    print("Save/Download this private output:", result_archive)
else:
    try:
        from google.colab import files
        files.download(str(result_archive))
    except ImportError:
        print("Download:", result_archive)


## Dữ liệu bổ sung và training: bước sau khi xem benchmark

Chưa khởi chạy retraining. Xem proposal trong tài liệu workflow: thu thập real hard cases (head box, cap/fur/sun hat, trẻ em, đầu nhỏ/che khuất, cảnh Việt Nam), annotate head box + helmet status + unknown, chia theo clip/location trước khi crop. Khóa test độc lập và baseline crop precision/recall/F1/AUROC/coverage cùng full-system metric trước khi quyết định training.

Sau khi xác định thành phần gây lỗi và xác minh dữ liệu/nhãn training, dùng **pipeline có sẵn** `scripts/prepare_probe_data.py` → `src/build_head_crops.py` → `src/train_probe.py` trên official training clips trong workspace riêng, không dùng held-out manifest này. `train_probe.py` hiện ghi đè `weights/dino_probe.joblib`; giữ probe baseline và provenance/SHA256, chỉ promote artifact sau khi so sánh cùng test. Notebook không có cell Run-all để retrain ngoài ý muốn.


## 8. Nhận diện ảnh/video trên T4 sau benchmark
Nếu đã chạy benchmark thành công, dùng lại phiên và model hiện có. Không chạy lại benchmark hoặc pip install. Chạy ô dưới, chọn code ZIP mới + job ZIP từ tab GPU Colab trong app local; tải ZIP kết quả rồi nhập lại tab đó. Giữ nguyên weights/model cache; code được cập nhật trước khi xử lý.


In [ ]:
#@title Chạy ảnh/video trên T4 trong phiên Colab đã cài model (không chạy lại benchmark)
from pathlib import Path
import json
import os
import stat
import subprocess
import sys
import uuid
from zipfile import ZipFile


def update_project_sources(archive_path, project):
    """Update trusted source ZIP; preserve all existing model/probe files."""
    project=Path(project).resolve()
    with ZipFile(archive_path) as archive:
        infos=archive.infolist()
        if len(infos)>500 or sum(i.file_size for i in infos)>16*1024**2:
            raise ValueError('Source ZIP unexpectedly large; use the small current code ZIP, not model/data archives.')
        if len({i.filename for i in infos})!=len(infos):raise ValueError('Duplicate ZIP members')
        targets=[]
        for entry in infos:
            name=entry.filename
            if '\\' in name or ':' in name or Path(name).is_absolute() or '..' in Path(name).parts:
                raise ValueError('Unsafe ZIP path')
            if stat.S_ISLNK(entry.external_attr>>16):raise ValueError('Source ZIP symlink rejected')
            target=(project/name).resolve()
            if not target.is_relative_to(project):raise ValueError('Unsafe source destination')
            if not entry.is_dir() and name.split('/')[0] in {'src','scripts','configs','weights'}:
                targets.append((entry,target))
        if 'scripts/run_gpu_job.py' not in archive.namelist():
            raise ValueError('Code ZIP is old or is a benchmark result. Download the current code ZIP from GPU Colab tab.')
        for entry,target in targets:
            if entry.filename.startswith('weights/') and target.exists():continue
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(archive.read(entry))


def run_colab_inference():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('This Python session has no CUDA. In Colab select the T4 GPU runtime; do not run this cell on local Intel.')
    from google.colab import files
    project=Path(globals().get('PROJECT','/content/helmet_runtime')).resolve()
    if not (project/'scripts/benchmark_runtime.py').is_file():
        raise RuntimeError('Use the Colab notebook/session where the benchmark ran, or run its setup/model cells first. No benchmark rerun is needed.')
    print('GPU:',torch.cuda.get_device_name(0))
    print('Select TWO ZIP files: current code ZIP + image/video request ZIP. Do not select helmet_runtime_benchmark.zip.')
    uploaded=files.upload()
    paths=[Path(name).resolve() for name in uploaded if name.lower().endswith('.zip')]
    del uploaded
    code=[];jobs=[]
    for path in paths:
        with ZipFile(path) as archive:
            names=set(archive.namelist())
            if 'scripts/run_gpu_job.py' in names:code.append(path)
            if 'request.json' in names:jobs.append(path)
    if len(code)!=1 or len(jobs)!=1:
        raise RuntimeError('Select one current code ZIP and one request ZIP. Benchmark ZIP contains measurements, not input media.')
    update_project_sources(code[0],project)
    # Preserve dependencies, downloaded models, HF_TOKEN and the active T4 runtime.
    os.chdir(project)
    if not os.environ.get('HF_TOKEN'):
        try:
            from google.colab import userdata
            token=userdata.get('HF_TOKEN')
            if token:os.environ['HF_TOKEN']=token
            del token
        except Exception:
            pass  # An already authorized model cache may suffice; runtime reports real access errors.
    output=project/'outputs/colab_gpu_notebook'/uuid.uuid4().hex
    subprocess.run([sys.executable,'scripts/run_gpu_job.py',str(jobs[0]),'--out',str(output)],cwd=project,check=True)
    results=list(output.glob('*/helmet_gpu_result_*.zip'))
    if len(results)!=1:raise RuntimeError('Expected one completed GPU result ZIP; inspect the preceding inference error.')
    with ZipFile(results[0]) as archive:metadata=json.loads(archive.read('result.json'))['result']
    print('Completed on GPU:',metadata['runtime'].get('cuda_device'))
    print('Media:',metadata['media'])
    print('Download this ZIP and import it in the local app GPU Colab tab:',results[0].name)
    files.download(str(results[0]))


if __name__=='__main__':run_colab_inference()
